EN ESTE NOTEBOOK:

*SUPERVISADO*
- Creamos 2 outcomes.
- Creamos 3 datasets con las siguientes variables de entrada: 1. Variables embarazo; 2. Variables clásicas de riesgo cv; 3. Combinado de los anteriores + otras variables.

*NO SUPERVISADO*
- Graficamos cada clúster marcando las variables con diferencias significativas (en cada cluster poner la variable con un color distinto).
- Dibujamos la mediana de cada clúster con una raya.
- Graficamos el score de riesgo Framingham (puntos + mediana).

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('datos.csv')

In [8]:
df2 = pd.read_csv('datos_preprocesados.csv')

# Outcome 1

riesgo_cv_compuesto = 1 si ≥1 de: 
1. TA sistólica ('ta_sistolica) ≥140 o TA diastólica ('ta_diastolica') ≥90 
2. Masa VI indexada ('masa_vi_tdiast_indexada') >95 g/m² 
3. Strain longitudinal VI ('strain_longitudinal_vi') > -16%
4. FEVI ('fevi_simpson_4c') < 50%
5. Cardiopatía estructural ('cardiopatia_estructural') presente

NOTA: Todas las variables están en datos.csv, excepto 'strain_longitudinal_vi', que está en datos_preprocesados.csv.

In [ ]:
# Definimos la columna clave y las variables del Outcome 1
ID_COL = 'id'
variables_outcome = [
    'ta_sistolica', 
    'ta_diastolica', 
    'masa_vi_tdiast_indexada', 
    'strain_longitudinal_vi', 
    'fevi_simpson_4c', 
    'cardiopatia_estructural'
]

# Hacemos el merge previo para tener todas las variables juntas
df_strain = df2[[ID_COL, 'strain_longitudinal_vi']]
df_total = pd.merge(df, df_strain, on=ID_COL, how='left')

In [ ]:
# Subconjunto solo con las variables que nos interesan para el análisis
df_analisis = df_total[variables_outcome]

print("ANÁLISIS DE MISSINGS POR VARIABLE")
# Calculamos la cantidad y el porcentaje de faltantes por cada variable
faltantes_por_variable = pd.DataFrame({
    'Valores Faltantes': df_analisis.isnull().sum(),
    'Porcentaje (%)': (df_analisis.isnull().mean() * 100).round(2)
}).sort_values(by='Porcentaje (%)', ascending=False)

display(faltantes_por_variable)

print("ANÁLISIS DE MISSINGS POR PACIENTE")

# Calculamos cuántas variables faltan por cada paciente (fila)
faltantes_por_paciente = df_analisis.isnull().sum(axis=1)

# Caso A: Pacientes que tienen TODO faltante (las 6 variables en NaN)
total_variables = len(variables_outcome)
pacientes_todo_faltante = (faltantes_por_paciente == total_variables).sum()
pct_todo_faltante = (pacientes_todo_faltante / len(df_total)) * 100

# Caso B: Pacientes con datos perfectos (0 variables faltantes)
pacientes_completos = (faltantes_por_paciente == 0).sum()
pct_completos = (pacientes_completos / len(df_total)) * 100

# Creamos una pequeña tabla resumen para el análisis por paciente
resumen_pacientes = pd.DataFrame({
    'Condición del Paciente': [
        'Tienen TODAS las variables faltantes (NaN)', 
        'Tienen datos COMPLETOS (0 faltantes)',
        'Tienen AL MENOS UN dato faltante'
    ],
    'Nº de Pacientes': [
        pacientes_todo_faltante, 
        pacientes_completos,
        (faltantes_por_paciente > 0).sum()
    ],
    'Porcentaje (%)': [
        f"{pct_todo_faltante:.2f}%", 
        f"{pct_completos:.2f}%",
        f"{((faltantes_por_paciente > 0).sum() / len(df_total) * 100):.2f}%"
    ]
})

display(resumen_pacientes)

ANÁLISIS DE MISSINGS POR VARIABLE


,Valores Faltantes,Porcentaje (%)
strain_longitudinal_vi,424,92.37
cardiopatia_estructural,78,16.99
masa_vi_tdiast_indexada,73,15.90
fevi_simpson_4c,70,15.25
ta_sistolica,4,0.87
ta_diastolica,4,0.87


ANÁLISIS DE MISSINGS POR PACIENTE


,Condición del Paciente,Nº de Pacientes,Porcentaje (%)
0,Tienen TODAS las variables faltantes (NaN),3,0.65%
1,Tienen datos COMPLETOS (0 faltantes),27,5.88%
2,Tienen AL MENOS UN dato faltante,432,94.12%


In [16]:
# Definimos los grupos de variables
variables_completas = [
    'ta_sistolica', 'ta_diastolica', 'masa_vi_tdiast_indexada', 
    'strain_longitudinal_vi', 'fevi_simpson_4c', 'cardiopatia_estructural'
]

variables_sin_strain = [
    'ta_sistolica', 'ta_diastolica', 'masa_vi_tdiast_indexada', 
    'fevi_simpson_4c', 'cardiopatia_estructural'
]

# Función auxiliar para evaluar las condiciones del outcome
def evaluar_riesgo(dataframe, incluir_strain=True):
    condicion = (
        (dataframe['ta_sistolica'] >= 140) | 
        (dataframe['ta_diastolica'] >= 90) | 
        (dataframe['masa_vi_tdiast_indexada'] > 95) | 
        (dataframe['fevi_simpson_4c'] < 50) | 
        (dataframe['cardiopatia_estructural'] == 1)
    )
    if incluir_strain:
        condicion = condicion | (dataframe['strain_longitudinal_vi'] > -16)
    return condicion.astype(int)

# Función para imprimir la tabla de distribución
def mostrar_distribucion(dataframe, titulo):
    dist = dataframe['riesgo_cv_compuesto'].value_counts().reset_index()
    dist.columns = ['riesgo_cv_compuesto', 'Nº Pacientes']
    dist['%'] = (dist['Nº Pacientes'] / len(dataframe) * 100).round(2).astype(str) + '%'
    print(f"\n{titulo} (Total N = {len(dataframe)})")
    display(dist)

In [17]:
# CASO 1: Todas las pacientes (con missings)
df_caso1 = df_total.copy()
df_caso1['riesgo_cv_compuesto'] = evaluar_riesgo(df_caso1, incluir_strain=True)
mostrar_distribucion(df_caso1, "CASO 1: Todas las pacientes")


CASO 1: Todas las pacientes (Total N = 459)


,riesgo_cv_compuesto,Nº Pacientes,%
0,0,380,82.79%
1,1,79,17.21%


In [18]:
# CASO 2: Pacientes sin NINGÚN missing
# Filtramos para quedarnos solo con filas que tengan las 6 variables completas
df_caso2 = df_total.dropna(subset=variables_completas).copy()
df_caso2['riesgo_cv_compuesto'] = evaluar_riesgo(df_caso2, incluir_strain=True)
mostrar_distribucion(df_caso2, "CASO 2: Pacientes sin ninguna variable missing")


CASO 2: Pacientes sin ninguna variable missing (Total N = 27)


,riesgo_cv_compuesto,Nº Pacientes,%
0,0,17,62.96%
1,1,10,37.04%


In [19]:
# CASO 3: Sin 'strain' y sin missings en el resto
# Filtramos para quedarnos con filas completas en las 5 variables restantes
df_caso3 = df_total.dropna(subset=variables_sin_strain).copy()
df_caso3['riesgo_cv_compuesto'] = evaluar_riesgo(df_caso3, incluir_strain=False)
mostrar_distribucion(df_caso3, "CASO 3: Sin 'strain_longitudinal_vi' y sin missings")


CASO 3: Sin 'strain_longitudinal_vi' y sin missings (Total N = 369)


,riesgo_cv_compuesto,Nº Pacientes,%
0,0,312,84.55%
1,1,57,15.45%


# Outcome 2

riesgo_cv_compuesto = 1 si ≥2 de: 
1. TA sistólica ('ta_sistolica) ≥140 o TA diastólica ('ta_diastolica') ≥90 
2. Masa VI indexada ('masa_vi_tdiast_indexada') >95 g/m² 
3. Strain longitudinal VI ('strain_longitudinal_vi') > -16%
4. FEVI ('fevi_simpson_4c') < 50%
5. Cardiopatía estructural ('cardiopatia_estructural') presente

NOTA: Todas las variables están en datos.csv, excepto 'strain_longitudinal_vi', que está en datos_preprocesados.csv.

In [21]:
# Función auxiliar para evaluar el outcome (>= 2 variables alteradas)
def evaluar_riesgo_severo(dataframe, incluir_strain=True):
    # Creamos una lista con las series booleanas (True=1, False=0)
    condiciones = [
        (dataframe['ta_sistolica'] >= 140),
        (dataframe['ta_diastolica'] >= 90),
        (dataframe['masa_vi_tdiast_indexada'] > 95),
        (dataframe['fevi_simpson_4c'] < 50),
        (dataframe['cardiopatia_estructural'] == 1)
    ]
    
    if incluir_strain:
        condiciones.append(dataframe['strain_longitudinal_vi'] > -16)
        
    # Sumamos elemento a elemento cuántas condiciones son True (se convierten en 1 al sumar)
    conteo_alteraciones = sum(cond.astype(int) for cond in condiciones)
    
    # Retornamos 1 si tiene 2 o más variables alteradas, de lo contrario 0
    return (conteo_alteraciones >= 2).astype(int)

# Función para imprimir la tabla de distribución
def mostrar_distribucion(dataframe, titulo):
    dist = dataframe['riesgo_cv_severo'].value_counts().reset_index()
    dist.columns = ['riesgo_cv_severo', 'Nº Pacientes']
    dist['%'] = (dist['Nº Pacientes'] / len(dataframe) * 100).round(2).astype(str) + '%'
    print(f"\n{titulo} (Total N = {len(dataframe)})")
    display(dist)

In [22]:
# CASO 1: Todas las pacientes (con missings)
df_severo_caso1 = df_total.copy()
df_severo_caso1['riesgo_cv_severo'] = evaluar_riesgo_severo(df_severo_caso1, incluir_strain=True)
mostrar_distribucion(df_severo_caso1, "CASO 1: Todas las pacientes (Outcome >= 2)")


CASO 1: Todas las pacientes (Outcome >= 2) (Total N = 459)


,riesgo_cv_severo,Nº Pacientes,%
0,0,436,94.99%
1,1,23,5.01%


In [23]:
# CASO 2: Pacientes sin NINGÚN missing
df_severo_caso2 = df_total.dropna(subset=variables_completas).copy()
df_severo_caso2['riesgo_cv_severo'] = evaluar_riesgo_severo(df_severo_caso2, incluir_strain=True)
mostrar_distribucion(df_severo_caso2, "CASO 2: Pacientes sin ninguna variable missing (Outcome >= 2)")


CASO 2: Pacientes sin ninguna variable missing (Outcome >= 2) (Total N = 27)


,riesgo_cv_severo,Nº Pacientes,%
0,0,27,100.0%


In [24]:
# CASO 3: Sin 'strain' y sin missings en el resto
df_severo_caso3 = df_total.dropna(subset=variables_sin_strain).copy()
df_severo_caso3['riesgo_cv_severo'] = evaluar_riesgo_severo(df_severo_caso3, incluir_strain=False)
mostrar_distribucion(df_severo_caso3, "CASO 3: Sin 'strain_longitudinal_vi' y sin missings (Outcome >= 2)")


CASO 3: Sin 'strain_longitudinal_vi' y sin missings (Outcome >= 2) (Total N = 369)


,riesgo_cv_severo,Nº Pacientes,%
0,0,348,94.31%
1,1,21,5.69%


# Creación datasets

## 1. Variables del embarazo: predicción longitudinal

In [25]:
# DATASET 1: Variables de la gestación (presentes en datos.csv / df)
variables_gestacion = [
    ID_COL,
    'fur_pre',
    'edad_materna_gest',
    'tas_1tri',
    'tad_1tri',
    'fecha_eco_1tri',
    'eg_eco_1tri',
    'uterinas_p95_1tri',
    'plgf_1tri',
    'fecha_plgf_1tri',
    'eg_plgf_1tri',
    'unidades_plgf_1tri',
    'valor_plgf_1tri',
    'riesgo_pe_1tri',
    'tomo_durante_gest',
    'uterinas_p95_eco_2tri',
    'fecha_eco_2tri',
    'eg_eco_2tri',
    'deter_sflt1_plgf_gest',
    'fecha_ult_deter',
    'eg_deter_sflt1_plgf',
    'valor_sflt1',
    'valor_plgf',
    'ratio_sflt1_plgf'
]

# Creamos el dataset asegurando que existan en el dataframe original
df_embarazo = df[[col for col in variables_gestacion if col in df.columns]].copy()

## 2. Variables clásicas actuales: predicción transversal

IMC, tabaco actual, colesterol actual, edad actual

In [26]:
# DATASET 2: Variables clásicas actuales (Predicción transversal)
# Mapeo según el diccionario para el estado basal actual (post-parto):
# - IMC actual: 'imc_actual'
# - Tabaco actual: 'fuma_post'
# - Colesterol actual: 'colesterol_total'
# - Edad actual: 'edad_actual'
variables_clasicas = [
    ID_COL,
    'imc_actual',
    'fuma_post',
    'colesterol_total',
    'edad_actual'
]

# Creamos el dataset
df_clasico = df[[col for col in variables_clasicas if col in df.columns]].copy()

In [27]:
# Verificación de las dimensiones de los nuevos datasets
print(f"Dataset 1 (Embarazo) - Dimensiones: {df_embarazo.shape}")
print(f"Dataset 2 (Clásico)  - Dimensiones: {df_clasico.shape}")

Dataset 1 (Embarazo) - Dimensiones: (459, 13)
Dataset 2 (Clásico)  - Dimensiones: (459, 5)


## 3. Variables embarazo + clásicas actuales + otras